<!--
Projeto : Agente Especialista em GRC - RAG + LLM local sobre o COSO ERM 2017
Arquivo : main.ipynb
-->
# Agente Especialista em GRC &ndash; demonstração

**Universidade Presbiteriana Mackenzie** &ndash; Faculdade de Computação e Informática
Sistemas de Informação &ndash; Inteligência Artificial (7ºK, noite) &ndash; Prof. Dr. Leandro Zerbinatti

| Integrante | RA | E-mail |
|---|---|---|
| Guilherme Martins Silva | 10417140 | 10417140@mackenzista.com.br |
| Enzo Ribeiro | 10418262 | 10418262@mackenzista.com.br |
| Pedro Nogueira Ribeiro | 10324924 | 10324924@mackenzista.com.br |

**Síntese:** notebook principal do agente. Inicializa o pipeline completo (re-ranker, memória
conversacional, banco vetorial do COSO ERM 2017 e Llama 3.2 3B Instruct em 4 bits) e responde a
perguntas em linguagem natural, mostrando a perplexidade e os trechos (chunks) usados como fonte.

**Histórico de alterações**

| Data | Autor | Descrição |
|---|---|---|
| 2026-02-26 | Guilherme Martins Silva | Versão inicial (instalação, inicialização e uma pergunta de teste). |
| 2026-11-15 | Enzo Ribeiro | Cabeçalho padronizado; `pip install` substituído pelo `requirements.txt`; escolha do modelo por variável de ambiente e dos modelos auxiliares na CPU em GPUs com menos de 8 GB; demonstração da memória conversacional e da abstenção fora do escopo. |

## Como executar

1. `pip install -r requirements.txt` (veja o `README.md`).
2. Modelo gerador: o repositório oficial `meta-llama/Llama-3.2-3B-Instruct` exige aceite da licença e
   `hf auth login` (ver `MANUAL_DE_USO.pdf`). Sem isso, defina `LLAMA_MODEL_ID=unsloth/Llama-3.2-3B-Instruct`,
   um espelho público com os mesmos pesos.
3. Execute as células em ordem a partir da raiz do repositório.

In [ ]:
import os
import warnings

import torch

from llama.call_llama import CallLlama

warnings.filterwarnings("ignore")

MODEL_ID = os.environ.get("LLAMA_MODEL_ID", "meta-llama/Llama-3.2-3B-Instruct")

# Com menos de 8 GB de VRAM, embeddings e re-ranker rodam na CPU e a GPU fica só para o Llama
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
AUX_DEVICE = None if vram_gb >= 8 else "cpu"
print(f"Modelo: {MODEL_ID} | VRAM: {vram_gb:.1f} GB | modelos auxiliares em: {AUX_DEVICE or 'GPU'}")

In [ ]:
llm = CallLlama(model_id=MODEL_ID, aux_device=AUX_DEVICE, verbose=False)

### 1. Pergunta objetiva sobre o framework

In [ ]:
resposta = llm.invoke(
    user="Quantos e quais são os componentes que estruturam o framework (ERM)?",
    seed=42
)
print(resposta)
print("\nFontes (chunks):", llm.last_metrics["sources"])

### 2. Pergunta de acompanhamento (usa a memória conversacional)

In [ ]:
resposta = llm.invoke(
    user="Dentre esses componentes, qual trata da definição do apetite a risco e por quê?",
    seed=42
)
print(resposta)

### 3. Pergunta fora do escopo do COSO ERM (o agente deve se abster)

In [ ]:
resposta = llm.invoke(user="Qual é a taxa Selic definida na última reunião do Copom?", seed=42)
print(resposta)